# PaySim1 Experiment

This notebook explores the KaggleHub PaySim1 dataset, validates the schema, compares a logistic baseline with a boosted tree model, and logs the run to MLflow before promoting the working steps into `ml_pipeline/`.

In [ ]:
import json
from pathlib import Path

import kagglehub
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_pipeline.data_validation import validate_dataset

mlflow.set_experiment("ai-risk-manager-paysim1")

dataset_root = Path(kagglehub.dataset_download("ealaxi/paysim1"))
csv_path = dataset_root / "PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(csv_path)
validate_dataset(df)
df = df.copy()
df["label"] = df["isFraud"].astype(int)
df = df.sort_values("step").reset_index(drop=True)

train_end = int(len(df) * 0.7)
val_end = int(len(df) * 0.85)
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

feature_columns = [column for column in df.columns if column not in {"label", "isFraud", "isFlaggedFraud", "nameOrig", "nameDest"}]
categorical_columns = [column for column in feature_columns if df[column].dtype == "object"]
numeric_columns = [column for column in feature_columns if column not in categorical_columns]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_columns),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_columns),
    ]
)

X_train = train_df[feature_columns]
y_train = train_df["label"]
X_val = val_df[feature_columns]
y_val = val_df["label"]
X_test = test_df[feature_columns]
y_test = test_df["label"]

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
baseline.fit(X_train, y_train)
baseline_val_prob = baseline.predict_proba(X_val)[:, 1]

boosted = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=150)),
])
boosted.fit(X_train, y_train)
boosted_val_prob = boosted.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.05, 0.95, 19)
best_threshold = 0.5
best_cost = float("inf")
for threshold in thresholds:
    predictions = (boosted_val_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, predictions).ravel()
    cost = fp * 20 + fn * 500 + int(predictions.sum()) * 10
    if cost < best_cost:
        best_cost = cost
        best_threshold = float(threshold)

boosted_test_prob = boosted.predict_proba(X_test)[:, 1]
boosted_test_pred = (boosted_test_prob >= best_threshold).astype(int)
metrics = {
    "precision": precision_score(y_test, boosted_test_pred, zero_division=0),
    "recall": recall_score(y_test, boosted_test_pred, zero_division=0),
    "f1": f1_score(y_test, boosted_test_pred, zero_division=0),
    "pr_auc": average_precision_score(y_test, boosted_test_prob),
    "roc_auc": roc_auc_score(y_test, boosted_test_prob),
}

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
model_path = artifacts_dir / "risk_model.joblib"
threshold_path = artifacts_dir / "threshold.json"
metadata_path = artifacts_dir / "metadata.json"

import joblib
joblib.dump(boosted, model_path)
threshold_path.write_text(json.dumps({"threshold": best_threshold}, indent=2) + "\n", encoding="utf-8")
metadata_path.write_text(json.dumps({"model_version": "ai-risk-manager-v1"}, indent=2) + "\n", encoding="utf-8")

with mlflow.start_run(run_name="notebook_experiment"):
    mlflow.log_params({"model": "HistGradientBoostingClassifier", "threshold": best_threshold})
    mlflow.log_metrics({f"test_{key}": float(value) for key, value in metrics.items()})
    mlflow.sklearn.log_model(boosted, artifact_path="model")
    mlflow.log_artifact(str(model_path))
    mlflow.log_artifact(str(threshold_path))
    mlflow.log_artifact(str(metadata_path))

metrics